In [16]:
from tbparse import SummaryReader
import os
import matplotlib.pyplot as plt
from tueplots.constants.color import palettes


metrics = [
    "Loss/Encoder",
    "Loss/Critic",
    "Loss/Actor",
    "Reward/Average Step Reward",
    "Evaluation/Average Reward",
    "Evaluation/Reward Std Dev",
]
plotTittles = [
    "Encoder Loss",
    "Critic Loss",
    "Actor Loss (Filtered)",
    "Average Step Reward",
    "Evaluation Average Reward"
]
models = [
    "td7_all",
    "td7_all_new",
    "td7_all_big",
    "td7_crash",
    "td7_all_offensive",
    "td7_plain",
    "td7_puck_proximity",
    "td7_offensive_pressure_puck_proximity"
]

labels = {
    "td7_all": "PlainVsAll",
    "td7_all_new": "SelfPlayOP+PPVsAll",
    "td7_all_big": "PlainSelfPlay+OP+PPVsBots",
    "td7_crash": "OP+PPVsAllSelfPlayVsAll",
    "td7_all_offensive": "OPVsTeam",
    "td7_plain": "PlainVsBots",
    "td7_puck_proximity": "PPVsBots",
    "td7_offensive_pressure_puck_proximity": "OP+PPVsBots"
}

colors = {
    "td7_all": palettes.tue_plot[0],
    "td7_all_new": palettes.tue_plot[1],
    "td7_all_big": palettes.tue_plot[2],
    "td7_crash": palettes.tue_plot[3],
    "td7_all_offensive": palettes.tue_plot[4],
    "td7_plain": palettes.tue_plot[5],
    "td7_puck_proximity": palettes.tue_plot[6],
    "td7_offensive_pressure_puck_proximity": palettes.tue_plot[7]
}
modelsDir = "models/td7"
saveDir = "assets/plots/td7"


In [17]:
modelReader = {
    mode: SummaryReader(log_path=os.path.join(modelsDir, mode), event_types=set(["scalars"]))
    for mode in models
}

In [18]:
scalerModelReader = {
    modelName: modelReader.scalars
    for modelName, modelReader in modelReader.items()
}

In [19]:
allDF = {
    model : df.pivot(index="step", columns="tag", values="value").reset_index()
    for model, df in scalerModelReader.items()
}

In [20]:
avgRewardStep = {
    model : df[['step','Evaluation/Average Reward']].dropna(subset=['Evaluation/Average Reward'])
    for model, df in allDF.items()
}


In [ ]:
plt.figure(figsize=(12, 3))

for i, (key, df) in enumerate(avgRewardStep.items()):
    plt.plot(df['step'], df['Evaluation/Average Reward'], label=labels[key], color=colors[key])

plt.xlabel('Number of Training Steps')
plt.ylabel('Smoothed Mean Reward')
plt.legend(title='models',ncol=3, loc="lower right")
plt.grid()
plt.tight_layout()  
plt.savefig(os.path.join(saveDir,"hockey_smoothed_mean_reward_during_training.png"), dpi=300, bbox_inches='tight')
plt.show()

In [22]:
stdRewardStep = {
    model : df[['step','Evaluation/Reward Std Dev']].dropna(subset=['Evaluation/Reward Std Dev'])
    for model, df in allDF.items()
}


In [ ]:
plt.figure(figsize=(12, 3))

for i, (key, df) in enumerate(stdRewardStep.items()):
    plt.plot(df['step'], df['Evaluation/Reward Std Dev'], label=labels[key], color=colors[key])

plt.xlabel('Number of Training Steps')
plt.ylabel('Smoothed Std Reward')
plt.legend(title='models',ncol=3, loc="upper right")
plt.grid()
plt.tight_layout()
plt.savefig(os.path.join(saveDir,"hockey_standard_deviation_reward_during_training.png"), dpi=300, bbox_inches='tight')
plt.show()

In [24]:
lossEncoder = {
    model : df[['step','Loss/Encoder']].dropna(subset=['Loss/Encoder'])
    for model, df in allDF.items()
}
lossCritic = {
    model : df[['step','Loss/Critic']].dropna(subset=['Loss/Critic'])
    for model, df in allDF.items()
}
lossActor = {
    model : df[['step','Loss/Actor']].dropna(subset=['Loss/Actor'])
    for model, df in allDF.items()
}

In [25]:
import matplotlib.pyplot as plt

def removeOutliers(df, col):
    Q1 = df[col].quantile(0.25)  
    Q3 = df[col].quantile(0.75)  
    IQR = Q3 - Q1  
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]  

def smothSeries(series, window=10):
    return series.rolling(window=window, min_periods=1).mean()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

lossDicts = {
    "Loss/Encoder": lossEncoder,
    "Loss/Critic": lossCritic,
    "Loss/Actor": lossActor
}

lossTitleDicts = {
    "Loss/Encoder": "Encoder Loss",
    "Loss/Critic": "Critic Loss",
    "Loss/Actor": "Actor Loss"
}

fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharex=True, sharey=False)

for ax, (title, loss_dict) in zip(axes, lossDicts.items()):
    for key, df in loss_dict.items():
        if title in df.columns and "step" in df.columns:
            filtered_df = removeOutliers(df, title) 
            filtered_df = filtered_df.interpolate(method='linear').fillna(method='bfill') 
            smoothed_loss = smothSeries(filtered_df[title], window=400)

            ax.plot(filtered_df["step"], smoothed_loss, label=key, color=colors[key])

    ax.set_xlabel('Number of Training Steps')
    ax.set_title(lossTitleDicts[title])
    ax.grid(True)

axes[0].set_ylabel('Smoothed Loss (No Outliers)')
ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=6))
handles, _ = axes[0].get_legend_handles_labels()
fig.legend(handles, labels.values(), loc='lower center', ncol=len(labels), fontsize=8)
plt.tight_layout(rect=[0, 0.1, 1, 1])  
plt.savefig(os.path.join(saveDir,"hockey_loss_during_training.png"), dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

with open("league_results.json", "r") as file:
    data = json.load(file)

teams = set()
for match in data.keys():
    team1, team2 = match.split(" vs ")
    teams.add(team1)
    teams.add(team2)

teams = sorted(teams)
heatmapData = pd.DataFrame(index=teams, columns=teams, dtype=float)

for match, results in data.items():
    team1, team2 = match.split(" vs ")
    winRate = results.get("win_rate", 0.0)
    heatmapData.loc[team1, team2] = winRate
    heatmapData.loc[team2, team1] = 1 - winRate  

heatmapData.fillna(0, inplace=True)

plt.figure(figsize=(16, 14))  
ax = sns.heatmap(heatmapData, annot=True, cmap="YlGnBu", fmt=".3f", linewidths=0.5)

plt.xlabel("Opponent", fontsize=14, labelpad=20)  
plt.ylabel("Agent", fontsize=14, labelpad=10)

plt.xticks(rotation=45, ha="right", fontsize=12)

plt.tight_layout()
plt.subplots_adjust(bottom=0.25) 
plt.savefig(os.path.join(saveDir,"hockey_self_leauge.png"), dpi=300, bbox_inches='tight')

plt.show()


In [37]:
import matplotlib.pyplot as plt
from tueplots.constants.color import palettes


metrics = [
    "Loss/Encoder",
    "Loss/Critic",
    "Loss/Actor",
    "Reward/Average Step Reward",
    "Evaluation/Average Reward",
    "Evaluation/Reward Std Dev",
]
plotTittles = [
    "Encoder Loss",
    "Critic Loss",
    "Actor Loss (Filtered)",
    "Average Step Reward",
    "Evaluation Average Reward"
]

models = [
    "td7_half_cheetah"
]

labels = {
    "td7_half_cheetah": "GymCheetahExperiment"
}

colors = {
    "td7_half_cheetah": palettes.tue_plot[0],
}
modelsDir = "models/td7"
saveDir = "assets/plots/td7"

In [38]:
modelReader = {
    mode: SummaryReader(log_path=os.path.join(modelsDir, mode), event_types=set(["scalars"]))
    for mode in models
}

In [39]:
scalerModelReader = {
    modelName: modelReader.scalars
    for modelName, modelReader in modelReader.items()
}

In [40]:
allDF = {
    model : df.pivot(index="step", columns="tag", values="value").reset_index()
    for model, df in scalerModelReader.items()
}

In [41]:
avgRewardStep = {
    model : df[['step','Evaluation/Average Reward']].dropna(subset=['Evaluation/Average Reward'])
    for model, df in allDF.items()
}

In [ ]:
plt.figure(figsize=(12, 3))

for i, (key, df) in enumerate(avgRewardStep.items()):
    plt.plot(df['step'], df['Evaluation/Average Reward'], label=labels[key], color=colors[key])

plt.xlabel('Number of Training Steps')
plt.ylabel('Smoothed Mean Reward')
plt.legend(title='models',ncol=3, loc="lower right")
plt.grid()
plt.tight_layout()  
plt.savefig(os.path.join(saveDir,"half_cheetah_smoothed_mean_reward_during_training.png"), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
stdRewardStep = {
    model : df[['step','Evaluation/Reward Std Dev']].dropna(subset=['Evaluation/Reward Std Dev'])
    for model, df in allDF.items()
}

In [ ]:
plt.figure(figsize=(12, 3))

for i, (key, df) in enumerate(stdRewardStep.items()):
    plt.plot(df['step'], df['Evaluation/Reward Std Dev'], label=labels[key], color=colors[key])

plt.xlabel('Number of Training Steps')
plt.ylabel('Smoothed Std Reward')
plt.legend(title='models',ncol=3, loc="upper right")
plt.grid()
plt.tight_layout()
plt.savefig(os.path.join(saveDir,"half_cheetah_standard_deviation_reward_during_training.png"), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
lossEncoder = {
    model : df[['step','Loss/Encoder']].dropna(subset=['Loss/Encoder'])
    for model, df in allDF.items()
}
lossCritic = {
    model : df[['step','Loss/Critic']].dropna(subset=['Loss/Critic'])
    for model, df in allDF.items()
}
lossActor = {
    model : df[['step','Loss/Actor']].dropna(subset=['Loss/Actor'])
    for model, df in allDF.items()
}

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

lossDicts = {
    "Loss/Encoder": lossEncoder,
    "Loss/Critic": lossCritic,
    "Loss/Actor": lossActor
}

lossTitleDicts = {
    "Loss/Encoder": "Encoder Loss",
    "Loss/Critic": "Critic Loss",
    "Loss/Actor": "Actor Loss"
}

fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharex=True, sharey=False)

for ax, (title, loss_dict) in zip(axes, lossDicts.items()):
    for key, df in loss_dict.items():
        if title in df.columns and "step" in df.columns:
            filtered_df = removeOutliers(df, title) 
            filtered_df = filtered_df.interpolate(method='linear').fillna(method='bfill') 
            smoothed_loss = smothSeries(filtered_df[title], window=400)

            ax.plot(filtered_df["step"], smoothed_loss, label=key, color=colors[key])

    ax.set_xlabel('Number of Training Steps')
    ax.set_title(lossTitleDicts[title])
    ax.grid(True)

axes[0].set_ylabel('Smoothed Loss (No Outliers)')
ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=6))
handles, _ = axes[0].get_legend_handles_labels()
fig.legend(handles, labels.values(), loc='lower center', ncol=len(labels), fontsize=8)
plt.tight_layout(rect=[0, 0.1, 1, 1])  
plt.savefig(os.path.join(saveDir,"half_cheetah_loss_during_training.png"), dpi=300, bbox_inches='tight')
plt.show()